# InChI db EDA

## Load InChI dataset

Reads the pre‑computed InChI strings from `inchi_output.txt` (tab‑separated, one InChI per line) into a pandas DataFrame with a single column `"InChI"`.  

The `db.head()` call displays the first few rows to verify the data loaded correctly.  

In [24]:
import pandas as pd
import os
from tokenizers import ByteLevelBPETokenizer
from transformers import PreTrainedTokenizerFast

# Add the parent directory (repository root) to sys.path
script_dir = '.'   # current directory (where the notebook is running)
inchi_file = os.path.join(script_dir, "inchi_output.txt")
db = pd.read_csv(inchi_file, header=None, names=["InChI"], sep="\t")
db_smiles = pd.read_csv(os.path.join(script_dir, "smiles_chembl.smi"), header=None, names = ["SMILES", "ChEMBL_ID"], sep="\t")

db.head()
db_smiles.head()

,SMILES,ChEMBL_ID
0,CCO,CHEMBL545
1,C,CHEMBL17564
2,CO,CHEMBL14688
3,NCCS,CHEMBL602
4,NCCN,CHEMBL816


## Load ByteLevelBPETokenizer

Loads the pre‑trained tokenizer from the saved `vocab.json` and `merges.txt` files located in the `inchi_tokenizer/` subdirectory.  
The tokenizer was previously trained on a large InChI corpus and will be used to encode InChI strings into token IDs.

In [ ]:
from tokenizers import ByteLevelBPETokenizer
import os
# Load the pre-trained ByteLevelBPETokenizer from the specified vocab and merges files
tokenizer = ByteLevelBPETokenizer(
    vocab=os.path.join(script_dir, "inchi_tokenizer", "vocab.json"),
    merges=os.path.join(script_dir, "inchi_tokenizer", "merges.txt"))
print("Loaded ByteLevelBPETokenizer")

Loaded ByteLevelBPETokenizer


## Tokenize all InChI strings

In order ti be able to execute the analysis also on the tokenized strings parallel to the raw InChI, we pre-tokenize all InChI strings in our `main()` function.

This function runs the tokenizer over every InChI stored in the database (`db`).  
The pre‑trained **ByteLevelBPETokenizer** (loaded from `inchi_tokenizer/vocab.json` and `inchi_tokenizer/merges.txt`) encodes each InChI into a list of token IDs.  

The output is saved as `inchi_tokens.txt` (one line per InChI, space‑separated token IDs).  
Progress is printed every 100,000 entries.  

This is a **one‑time preprocessing step**; subsequent analysis (e.g., token length statistics, verification) reads the cached file rather than re‑encoding the entire corpus.  

The `main()` function is currently commented out in the script. Uncomment it if if is needed to regenerate the tokenized file.

In [ ]:
def main():
    output_file = os.path.join(script_dir, "inchi_tokens.txt")
    total = len(db)
    print(f"Starting tokenization of {total} InChIs...")
    with open(output_file, 'w', encoding='utf-8') as f_out:
        for i, inchi in enumerate(db["InChI"]):
            encoded = tokenizer.encode(inchi)
            token_ids = encoded.ids
            f_out.write(" ".join(map(str, token_ids)) + "\n")
            if (i + 1) % 100000 == 0:
                print(f"Processed {i+1}/{total} InChIs")
    print(f"Tokenization complete. Output saved to {output_file}")

with open("inchi_tokens.txt", "r") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.strip())

281 33 21 55 19 39 22 44 26 51 19 71 21 17 22 17 23 19 76 23 44 16 22 44 22 16 21 44 23
281 33 21 55 19 783 24 19 76 21 44 24
281 33 21 55 19 783 24 51 19 71 21 17 22 19 76 22 44 16 21 44 23
281 33 21 55 19 39 22 44 27 582 19 71 23 17 21 17 22 17 24 19 76 24 44 16 21 17 23 44 22
281 33 21 55 19 39 22 44 28 50 22 19 71 23 17 21 17 22 17 24 19 76 21 17 24 44 22


## Max InChI str length

In [12]:
# --- 1. Max raw string length ---
max_str_len = max(db["InChI"].apply(len))
print(f"Longest InChI (characters): {max_str_len}")

Longest InChI (characters): 3915


## Tokenize longest InChI

In [13]:
# --- 2. Find the InChI with maximum string length and tokenize it ---
# Get the actual string
longest_inchi = db.loc[db["InChI"].str.len().idxmax(), "InChI"]

# Tokenize
encoded = tokenizer.encode(longest_inchi)
token_len = len(encoded.ids)
print(f"Longest InChI tokenized length: {token_len} tokens")
print(f"Sample tokens: {encoded.tokens[:15]}...")  # show first 15

Longest InChI tokenized length: 2068 tokens
Sample tokens: ['InChI', '=', '1', 'S', '/', 'C', '325', 'H', '387', 'N', '118', 'O', '209', 'P', '29']...


## text


In [16]:
import numpy as np

# --- 3. Compute token length percentiles ---
def compute_token_length_percentiles():
    tokens_file = os.path.join(script_dir, "inchi_tokens.txt")

    lengths = []
    with open(tokens_file, 'r') as f:
        print(f"Computing token length statistics from {tokens_file}...")
        for i, line in enumerate(f):
            token_count = len(line.split())
            lengths.append(token_count)

    # Convert to numpy array for percentile calculation
    lengths_np = np.array(lengths)
    percentiles = [1, 50, 75, 90, 99, 99.9]
    p = np.percentile(lengths_np, percentiles)
    print(f"\n--- Token Length Statistics ---")
    print(f"Total sequences: {len(lengths)}")
    print(f"Mean length: {np.mean(lengths_np):.2f}")
    print(f"Median: {np.median(lengths_np)}")
    print(f"Max length: {np.max(lengths_np)}")
    print(f"Min length: {np.min(lengths_np)}")
    for perc, val in zip(percentiles, p):
        print(f"{perc}th percentile: {val}")

compute_token_length_percentiles()

Computing token length statistics from .\inchi_tokens.txt...

--- Token Length Statistics ---
Total sequences: 1575727
Mean length: 109.74
Median: 102.0
Max length: 2068
Min length: 11
1th percentile: 55.0
50th percentile: 102.0
75th percentile: 119.0
90th percentile: 142.0
99th percentile: 305.0
99.9th percentile: 844.0


In [20]:
def analyze_duplicates(df):
    duplicates = df.duplicated(subset="InChI").sum()
    total = len(df)
    print(f"\n--- Duplicate Analysis ---")
    print(f"Duplicate molecules (by InChI): {duplicates}")
    print(f"Percentage of duplicates: {duplicates / total * 100:.4f}%")

analyze_duplicates(db)


--- Duplicate Analysis ---
Duplicate molecules (by InChI): 73565
Percentage of duplicates: 4.6686%


In [25]:
dup_inchis = db["InChI"].value_counts()
dup_smiles = db_smiles["SMILES"].value_counts()
print((dup_inchis > 1).sum())  # count of InChIs with multiple entries
print(max(dup_inchis))  # max duplicates for a single InChI
print((dup_smiles > 1).sum())  # count of SMILES with multiple entries
print(max(dup_smiles))  # max duplicates for a single SMILES

54689
510
54472
510


In [29]:
def analyze_inchi_structure(inchi):
    return {
        "length": len(inchi),
        "num_layers": inchi.count("/"),
        "has_stereo": int("/t" in inchi or "/m" in inchi or "/b" in inchi or "/s" in inchi),
    }
total = len(db)
num_layers = 0
has_stereo_count = 0
for i in db["InChI"]:
    struct = analyze_inchi_structure(i)
    num_layers += struct["num_layers"]
    has_stereo_count += struct["has_stereo"]

print(f"\n--- InChI Structural Analysis ---")
print(f"Average number of layers: {num_layers / total:.2f}")
print(f"Percentage of InChIs with stereochemistry: {has_stereo_count / total * 100:.2f}%")
print(f"Total stereochemistry entries: {has_stereo_count}")




--- InChI Structural Analysis ---
Average number of layers: 3.01
Percentage of InChIs with stereochemistry: 0.01%
Total stereochemistry entries: 93


NO hay stereoquimica ya que \ en los string de smiles dan error por escape character (esto es para estereiquimica de doble bond trans) mientras que si que hay para los cis ya que usan solo / y no \. Tampoco hay R o S ya que ninguna SMILES usa @. Bias en contra de stereoquimica, potencial caveat. 

In [30]:
from collections import Counter

def token_frequency(inchi_tokens_file):
    counter = Counter()
    with open(inchi_tokens_file) as f:
        for line in f:
            counter.update(line.split())
    
    print(f"Unique tokens: {len(counter)}")
    print("Top 20 tokens:", counter.most_common(20))

token_frequency(os.path.join(script_dir, "inchi_tokens.txt"))

Unique tokens: 833
Top 20 tokens: [('17', 34773281), ('16', 11058201), ('13', 10963604), ('12', 10011912), ('44', 7377962), ('22', 4871294), ('21', 4757090), ('19', 4739861), ('23', 4272177), ('24', 2777017), ('264', 2512011), ('263', 2504137), ('265', 2493096), ('261', 2491681), ('25', 2479527), ('266', 2463497), ('267', 2457925), ('262', 2422508), ('268', 2413650), ('29', 2365482)]


## Verify tokenizer determinism

This function performs a **random‑sampling verification** to ensure that the cached token IDs (`inchi_tokens.txt`) are consistent with the tokenizer’s current behavior.  

- It selects a random InChI from the dataset, retrieves its saved token IDs, and re‑encodes the same InChI using the tokenizer.  
- The two sets of token IDs are compared; if they match, the tokenizer is confirmed to produce deterministic output and the cached file is correctly aligned.  

This check addresses the inherent “black‑box” nature of tokenization pipelines, providing an empirical guarantee that the pre‑processed data can be reliably reused without silent inconsistencies.

In [9]:
import random
# --- Verification: compare a random line from saved file with fresh encoding ---
def verify_saved_tokens_random():
    tokens_file = os.path.join(script_dir, "inchi_tokens.txt")
    if not os.path.exists(tokens_file):
        print(f"Error: {tokens_file} not found.")
        return

    # Read all lines to count them and pick a random line
    with open(tokens_file, 'r') as f:
        lines = f.readlines()
    total_lines = len(lines)

    # Pick a random line number (1‑based)
    line_number = random.randint(1, total_lines)
    saved_ids = list(map(int, lines[line_number-1].strip().split()))

    # Get the original InChI from the dataframe (0‑based index)
    original_inchi = db.iloc[line_number-1]["InChI"]

    # Freshly encode the original InChI
    fresh_encoded = tokenizer.encode(original_inchi)
    fresh_ids = fresh_encoded.ids

    # Decode the saved IDs back to a string
    decoded_str = tokenizer.decode(saved_ids)

    print(f"=== Verification for random line {line_number} (of {total_lines}) ===")
    print(f"Original InChI        : {original_inchi}")
    print(f"Decoded from saved IDs: {decoded_str}")
    print(f"Token IDs match       : {saved_ids == fresh_ids}")

    if saved_ids != fresh_ids:
        print("\nMismatch details:")
        print(f"Saved IDs : {saved_ids}")
        print(f"Fresh IDs : {fresh_ids}")
        print(f"Saved tokens : {tokenizer.decode(saved_ids, skip_special_tokens=False)}")
        print(f"Fresh tokens : {tokenizer.decode(fresh_ids, skip_special_tokens=False)}")
    else:
        print("Verification passed: saved tokens match fresh encoding.")

verify_saved_tokens_random()

=== Verification for random line 673089 (of 1575727) ===
Original InChI        : InChI=1S/C19H24N2O6/c1-4-25-18(23)17-13(3)20-19(24)21-14(17)11-27-16(22)9-10-26-15-8-6-5-7-12(15)2/h5-8,13H,4,9-11H2,1-3H3,(H2,20,21,24)
Decoded from saved IDs: InChI=1S/C19H24N2O6/c1-4-25-18(23)17-13(3)20-19(24)21-14(17)11-27-16(22)9-10-26-15-8-6-5-7-12(15)2/h5-8,13H,4,9-11H2,1-3H3,(H2,20,21,24)
Token IDs match       : True
Verification passed: saved tokens match fresh encoding.
